# SingleTaskVariationalGP

この Notebook では、Exact GP の学習が重くなるデータ量に対して近似 GP として利用できる `robotorchan.models.SingleTaskVariationalGP` を扱います。

inducing points と `VariationalELBO` を用いた minibatch 学習を実装します。

## 1. このモデルを使う場面

Exact GP の計算量が実用上のボトルネックになる場合に variational GP を検討します。近似精度と計算量は主に inducing points の数と配置に依存します。

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

from robotorchan.models import SingleTaskVariationalGP

torch.manual_seed(0)
dtype = torch.double

## 2. 合成学習データ

In [ ]:
def objective(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x) + 0.25 * torch.cos(6.0 * torch.pi * x)

n_train = 400
train_X = torch.rand(n_train, 1, dtype=dtype)
train_Y = objective(train_X) + 0.08 * torch.randn(n_train, 1, dtype=dtype)

train_X.shape, train_Y.shape

## 3. モデル構築

ここでは 400 点の学習データに対して 48 個の inducing points を使用します。

In [ ]:
model = SingleTaskVariationalGP(
    train_X=train_X,
    train_Y=train_Y,
    inducing_points=48,
)
model

## 4. robotorchan 共通 API

wrapper はコンストラクタに渡したテンソルを保持し、ELBO を生成する `make_mll()` を提供します。上流の variational constructor には `train_Yvar` がないため、`raw_train_Yvar` は `None` です。

In [ ]:
print("raw_train_X:", model.raw_train_X.shape)
print("raw_train_Y:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data_names:", model.raw_data_names)
print("supports_mll:", model.supports_mll)

mll = model.make_mll(num_data=n_train)
type(mll).__name__

## 5. Minibatch による variational 学習

各 ELBO 評価では minibatch だけを使いますが、`num_data` には学習データ全体の件数を指定します。学習対象となる近似 GP は `model.model`、likelihood は `model.likelihood` から利用できます。

In [ ]:
loader = DataLoader(
    TensorDataset(train_X, train_Y),
    batch_size=64,
    shuffle=True,
)

model.train()
model.likelihood.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
mll = model.make_mll(num_data=n_train)

loss_history = []
for epoch in range(60):
    epoch_loss = 0.0
    for batch_X, batch_Y in loader:
        optimizer.zero_grad()
        output = model.model(batch_X)
        loss = -mll(output, batch_Y.squeeze(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history.append(epoch_loss / len(loader))

print(f"initial loss: {loss_history[0]:.3f}")
print(f"final loss:   {loss_history[-1]:.3f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Negative ELBO")
plt.title("Variational training history")
plt.show()

## 6. Posterior 予測

In [ ]:
model.eval()
model.likelihood.eval()

test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

lower = mean - 1.96 * std
upper = mean + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), s=8, alpha=0.25, label="observations")
plt.plot(test_X.squeeze(-1), objective(test_X).squeeze(-1), linestyle="--", label="true function")
plt.plot(test_X.squeeze(-1), mean, label="posterior mean")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("SingleTaskVariationalGP posterior")
plt.show()

## 7. Exact GP との使い分け

小規模データでは exact inference が単純で統計的にも効率的な `SingleTaskGP` が通常は第一候補です。データ量が増えて covariance の分解が重くなると `SingleTaskVariationalGP` が有力になります。inducing points の数は計算速度と近似精度のトレードオフを制御します。